In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', None)

# Paths to annotator files
DATA_DIR = "data_received/task_a"
FILE_A01 = os.path.join(DATA_DIR, "task_a_A01.csv")
FILE_A02 = os.path.join(DATA_DIR, "task_a_A02.csv")
FILE_TEXT = "data_processed/task_a/task_a_web_annotation_240.csv"

# Load datasets
df_a01 = pd.read_csv(FILE_A01)
df_a02 = pd.read_csv(FILE_A02)
df_text = pd.read_csv(FILE_TEXT) if os.path.exists(FILE_TEXT) else None

print(f"Loaded A01 annotations: {df_a01.shape[0]} rows, {df_a01.shape[1]} columns")
print(f"Loaded A02 annotations: {df_a02.shape[0]} rows, {df_a02.shape[1]} columns")
if df_text is not None:
    print(f"Loaded source metadata: {len(df_text)} items for context mapping.")


Loaded A01 annotations: 240 rows, 17 columns
Loaded A02 annotations: 240 rows, 17 columns
Loaded source metadata: 240 items for context mapping.


In [2]:
# 1. Verify sample counts and sample_id uniqueness
assert len(df_a01) == 240, f"A01 must have 240 rows, actual: {len(df_a01)}"
assert len(df_a02) == 240, f"A02 must have 240 rows, actual: {len(df_a02)}"
assert df_a01['sample_id'].nunique() == 240, "Duplicate sample_id found in A01!"
assert df_a02['sample_id'].nunique() == 240, "Duplicate sample_id found in A02!"

# 2. Check 1-to-1 sample_id correspondence
set_a01 = set(df_a01['sample_id'])
set_a02 = set(df_a02['sample_id'])
assert set_a01 == set_a02, "sample_id mismatch between A01 and A02!"

# 3. Merge on sample_id
merged_df = pd.merge(
    df_a01[['sample_id', 'human_toxicity_label']],
    df_a02[['sample_id', 'human_toxicity_label']],
    on='sample_id',
    suffixes=('_a01', '_a02')
)
print("Validation passed: 240 aligned unique samples between A01 and A02.")


Validation passed: 240 aligned unique samples between A01 and A02.


In [3]:
# Extract disagreements between A01 and A02
y_a01 = merged_df['human_toxicity_label_a01'].astype('string').str.strip().str.lower()
y_a02 = merged_df['human_toxicity_label_a02'].astype('string').str.strip().str.lower()
diff_mask = (y_a01 != y_a02)
disagree_df = merged_df[diff_mask].copy()

if 'text' not in disagree_df.columns:
    df_raw = pd.read_csv("data_processed/task_a/task_a_web_annotation_240.csv")
    id_col = 'id' if 'id' in df_raw.columns else 'sample_id'
    disagree_df = pd.merge(disagree_df, df_raw[[id_col, 'text']], left_on='sample_id', right_on=id_col, how='left')

disagree_df['A01_label'] = disagree_df['human_toxicity_label_a01']
disagree_df['A02_label'] = disagree_df['human_toxicity_label_a02']

simple_diff_table = disagree_df[['sample_id', 'text', 'A01_label', 'A02_label']]
print(f"Total disagreements: {len(simple_diff_table)} out of {len(merged_df)} items ({len(simple_diff_table)/len(merged_df)*100:.1f}%)")


Total disagreements: 66 out of 240 items (27.5%)


In [4]:
# Save disagreements file for third annotator adjudication
output_filename = "data_received/task_a/disagreements_task_a.csv"
print(f"Disagreements file: {output_filename}")


Disagreements file: data_received/task_a/disagreements_task_a.csv


In [5]:
import os
import pandas as pd

# 1. File paths
DATA_DIR = "data_received/task_a"
FILE_A01 = os.path.join(DATA_DIR, "task_a_A01.csv")
FILE_A02 = os.path.join(DATA_DIR, "task_a_A02.csv")
FILE_TEXT = "data_processed/task_a/task_a_web_annotation_240.csv"

# 2. Load data
df_a01 = pd.read_csv(FILE_A01)
df_a02 = pd.read_csv(FILE_A02)
df_text = pd.read_csv(FILE_TEXT) if os.path.exists(FILE_TEXT) else None

# 3. Extract agreed items (A01 == A02)
merged = pd.merge(df_a01, df_a02, on="sample_id", suffixes=("_a01", "_a02"))
agree_df = merged[merged["human_toxicity_label_a01"] == merged["human_toxicity_label_a02"]].copy()
print(f"Agreed items (A01 == A02): {len(agree_df)} / 240")


Agreed items (A01 == A02): 174 / 240


In [6]:
import pandas as pd
from collections import Counter

# Load disagreements file
file_path = "data_received/task_a/disagreements_task_a.csv"
df_diff = pd.read_csv(file_path, keep_default_na=False)

def check_3_labels(row):
    labels = [
        str(row['A01_label']).strip().lower(),
        str(row['A02_label']).strip().lower(),
        str(row['A03_label']).strip().lower()
    ]
    counts = Counter(labels)
    val, count = counts.most_common(1)[0]
    return val if count >= 2 else "TIE"

df_diff['majority_label'] = df_diff.apply(check_3_labels, axis=1)
print(f"Resolved {len(df_diff)} disputed items via 3rd annotator (A03).")


Resolved 66 disputed items via 3rd annotator (A03).


In [7]:
import os
import pandas as pd
from collections import Counter

# Load adjudicated disagreements
file_diff = "data_received/task_a/disagreements_task_a.csv"
df_diff = pd.read_csv(file_diff, keep_default_na=False)
print(f"Adjudicated disagreements loaded: {len(df_diff)} items.")


Adjudicated disagreements loaded: 66 items.


In [8]:
import os
import pandas as pd
from collections import Counter

# 1. File paths
DATA_DIR = "data_received/task_a"
FILE_A01 = os.path.join(DATA_DIR, "task_a_A01.csv")
FILE_A02 = os.path.join(DATA_DIR, "task_a_A02.csv")
FILE_DIS = os.path.join(DATA_DIR, "disagreements_task_a.csv")
FILE_META = "data_processed/task_a/task_a_web_annotation_240.csv"

# 2. Load data
df_a01 = pd.read_csv(FILE_A01)
df_a02 = pd.read_csv(FILE_A02)
try:
    df_dis = pd.read_csv(FILE_DIS, encoding='cp1252')
except Exception:
    df_dis = pd.read_csv(FILE_DIS, encoding='utf-8')
df_meta = pd.read_csv(FILE_META)

# 3. Map annotations and metadata
threshold_gold = pd.DataFrame()
threshold_gold['sample_id'] = df_meta['id']
threshold_gold['text'] = df_meta['text']
threshold_gold['original'] = df_meta['label']

map_a01 = dict(zip(df_a01['sample_id'], df_a01['human_toxicity_label']))
map_a02 = dict(zip(df_a02['sample_id'], df_a02['human_toxicity_label']))
map_a03 = dict(zip(df_dis['sample_id'], df_dis['A03_label']))

threshold_gold['labelA01'] = threshold_gold['sample_id'].map(map_a01)
threshold_gold['labelA02'] = threshold_gold['sample_id'].map(map_a02)
threshold_gold['labelA03'] = threshold_gold['sample_id'].map(map_a03).fillna("")

# 4. Resolve gold label (majority >= 2 votes, otherwise unsure)
def calc_gold(row):
    l1 = str(row['labelA01']).strip().lower()
    l2 = str(row['labelA02']).strip().lower()
    l3 = str(row['labelA03']).strip().lower()
    
    if l1 == l2 and l1 in ['toxic', 'nontoxic']:
        return l1
    labels = [l for l in [l1, l2, l3] if l in ['toxic', 'nontoxic']]
    counts = Counter(labels)
    if counts['toxic'] >= 2:
        return 'toxic'
    elif counts['nontoxic'] >= 2:
        return 'nontoxic'
    else:
        return 'unsure'

threshold_gold['gold_label'] = threshold_gold.apply(calc_gold, axis=1)

# Add metadata columns
meta_cols = ['subtype', 'topic_type', 'scenario', 'industry', 'register', 'score', 'band', 'off_target']
for col in meta_cols:
    if col in df_meta.columns:
        threshold_gold[col] = df_meta[col]

# 5. Save consolidated dataset to data_human_gold_label/task_a_goldlabel_240.csv
gold_dir = "data_human_gold_label"
os.makedirs(gold_dir, exist_ok=True)
gold_out_path = os.path.join(gold_dir, "task_a_goldlabel_240.csv")
threshold_gold.to_csv(gold_out_path, index=False, encoding="utf-8-sig")

print(f"Generated and saved: {gold_out_path} ({len(threshold_gold)} rows)")
print(threshold_gold['gold_label'].value_counts())


Generated and saved: data_human_gold_label\task_a_goldlabel_240.csv (240 rows)
gold_label
nontoxic    136
toxic       103
unsure        1
Name: count, dtype: int64


In [9]:
import os
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import roc_auc_score

# 1. Load dataset
df = pd.read_csv("data_human_gold_label/task_a_goldlabel_240.csv", keep_default_na=False)

# 2. Load model
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_ID = "unitary/unbiased-toxic-roberta"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID).to(DEVICE)
model.eval()

id2label = model.config.id2label
TOX_IDX = next((int(i) for i, l in id2label.items() if str(l).lower() in {'toxic', 'toxicity'}), 0)

@torch.no_grad()
def predict_toxicity_scores(texts, batch_size=32):
    all_scores = []
    text_list = list(texts)
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=256).to(DEVICE)
        probs = torch.sigmoid(model(**inputs).logits.float())[:, TOX_IDX]
        all_scores.extend(probs.cpu().numpy().tolist())
    return np.array(all_scores)

df['model_toxic_prob'] = predict_toxicity_scores(df['text'].fillna("").tolist())

# 3. Calculate AUC vs Prompted Original (240 samples)
orig_col = 'original' if 'original' in df.columns else 'label'
y_orig_true = df[orig_col].str.strip().str.lower().map({'nontoxic': 0, 'toxic': 1})
auc_orig = roc_auc_score(y_orig_true, df['model_toxic_prob'])

# 4. Calculate AUC vs Human Gold Label (excluding 1 unsure -> 239 samples)
df_gold = df[df['gold_label'].str.strip().str.lower().isin(['toxic', 'nontoxic'])].copy()
n_removed_unsure = len(df) - len(df_gold)
y_gold_true = df_gold['gold_label'].str.strip().str.lower().map({'nontoxic': 0, 'toxic': 1})
auc_gold = roc_auc_score(y_gold_true, df_gold['model_toxic_prob'])

comparison_df = pd.DataFrame([
    {
        "Evaluation Reference": "Prompted Label (LLM)",
        "Total Samples": len(df),
        "Toxic Count": int((y_orig_true == 1).sum()),
        "Non-toxic Count": int((y_orig_true == 0).sum()),
        "AUC-ROC Score": round(auc_orig, 4)
    },
    {
        "Evaluation Reference": f"Human Gold Label (Excl. {n_removed_unsure} unsure)",
        "Total Samples": len(df_gold),
        "Toxic Count": int((y_gold_true == 1).sum()),
        "Non-toxic Count": int((y_gold_true == 0).sum()),
        "AUC-ROC Score": round(auc_gold, 4)
    }
])
print(comparison_df.to_string(index=False))
print(f"\nAUC prompted {auc_orig:.4f} ≈ {auc_orig:.3f}, human {auc_gold:.4f} ≈ {auc_gold:.3f}")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: unitary/unbiased-toxic-roberta
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


             Evaluation Reference  Total Samples  Toxic Count  Non-toxic Count  AUC-ROC Score
             Prompted Label (LLM)            240          175               65         0.5874
Human Gold Label (Excl. 1 unsure)            239          103              136         0.5595

AUC prompted 0.5874 ≈ 0.587, human 0.5595 ≈ 0.559


In [10]:
from sklearn.metrics import cohen_kappa_score

# 1. Filter out unsure (239 binary samples)
df_clean_239 = df[df['gold_label'].str.strip().str.lower().isin(['toxic', 'nontoxic'])].copy()
orig_col = 'original' if 'original' in df_clean_239.columns else 'label'

y_prompt = df_clean_239[orig_col].str.strip().str.lower()
y_human = df_clean_239['gold_label'].str.strip().str.lower()

# 2. Calculate counts dynamically
A = int(((y_prompt == 'toxic') & (y_human == 'toxic')).sum())
B = int(((y_prompt == 'toxic') & (y_human == 'nontoxic')).sum())
C = int(((y_prompt == 'nontoxic') & (y_human == 'toxic')).sum())
D = int(((y_prompt == 'nontoxic') & (y_human == 'nontoxic')).sum())

n_total = len(df_clean_239)
n_agree = A + D
n_disagree = B + C
pct_agree = (n_agree / n_total) * 100.0
kappa = cohen_kappa_score(y_prompt, y_human, labels=['toxic', 'nontoxic'])

# 3. Contingency table with margins
cross_table = pd.DataFrame(
    data=[
        [A, B, A + B],
        [C, D, C + D],
        [A + C, B + D, n_total]
    ],
    index=["Toxic", "Non-toxic", "Total"],
    columns=["Toxic", "Non-toxic", "Total"]
)
cross_table.index.name = "Prompted label"
cross_table.columns.name = "Human label"

print(cross_table)
print(f"\n{B}/{A + B} prompted-toxic evaluated as non-toxic by human annotators")
print(f"Agreement {n_agree}/{n_total}, Cohen’s κ {kappa:.4f} ≈ {kappa:.3f}")


Human label     Toxic  Non-toxic  Total
Prompted label                         
Toxic              97         77    174
Non-toxic           6         59     65
Total             103        136    239

77/174 prompted-toxic evaluated as non-toxic by human annotators
Agreement 156/239, Cohen’s κ 0.3466 ≈ 0.347


In [11]:
# Subtype Detailed Evaluation
TAU = 0.5

def evaluate_subtypes_complete(data, label_col):
    rows = []
    valid_data = data[data[label_col].str.strip().str.lower().isin(['toxic', 'nontoxic'])].copy()
    valid_data['y_num'] = (valid_data[label_col].str.strip().str.lower() == 'toxic').astype(int)
    
    all_nontoxic = valid_data[valid_data['y_num'] == 0]
    all_toxic = valid_data[valid_data['y_num'] == 1]
    subtypes = sorted(valid_data['subtype'].dropna().unique())
    
    for st in subtypes:
        sub_df = valid_data[valid_data['subtype'] == st]
        y_true_sub = sub_df['y_num'].values
        y_score_sub = sub_df['model_toxic_prob'].values
        y_pred_sub = (y_score_sub >= TAU).astype(int)
        
        n_total = len(sub_df)
        n_toxic = (y_true_sub == 1).sum()
        n_nontoxic = (y_true_sub == 0).sum()
        
        tp = ((y_true_sub == 1) & (y_pred_sub == 1)).sum()
        fn = ((y_true_sub == 1) & (y_pred_sub == 0)).sum()
        tn = ((y_true_sub == 0) & (y_pred_sub == 0)).sum()
        fp = ((y_true_sub == 0) & (y_pred_sub == 1)).sum()
        
        rec_toxic = (tp / (tp + fn)) if (tp + fn) > 0 else np.nan
        prec_toxic = (tp / (tp + fp)) if (tp + fp) > 0 else np.nan
        rec_nontoxic = (tn / (tn + fp)) if (tn + fp) > 0 else np.nan
        prec_nontoxic = (tn / (tn + fn)) if (tn + fn) > 0 else np.nan
        
        if len(np.unique(y_true_sub)) > 1:
            auc_val = roc_auc_score(y_true_sub, y_score_sub)
        else:
            if n_toxic > 0:
                eval_set = pd.concat([sub_df, all_nontoxic]).drop_duplicates(subset=['sample_id'])
            else:
                eval_set = pd.concat([sub_df, all_toxic]).drop_duplicates(subset=['sample_id'])
            auc_val = roc_auc_score(eval_set['y_num'], eval_set['model_toxic_prob'])
            
        rows.append({
            "Subtype": st,
            "Total": n_total,
            "N_Toxic": n_toxic,
            "N_NonToxic": n_nontoxic,
            "Recall_Toxic": f"{rec_toxic*100:.1f}%" if not np.isnan(rec_toxic) else "-",
            "Prec_Toxic": f"{prec_toxic*100:.1f}%" if not np.isnan(prec_toxic) else "-",
            "Recall_NonToxic": f"{rec_nontoxic*100:.1f}%" if not np.isnan(rec_nontoxic) else "-",
            "Prec_NonToxic": f"{prec_nontoxic*100:.1f}%" if not np.isnan(prec_nontoxic) else "-",
            "AUC_ROC": round(auc_val, 4)
        })
        
    return pd.DataFrame(rows)

orig_col = 'original' if 'original' in df.columns else 'label'
print(evaluate_subtypes_complete(df, 'gold_label'))


                             Subtype  Total  N_Toxic  N_NonToxic Recall_Toxic  \
0              belittling_competence     25       18           7        11.1%   
1                           contempt     25       13          12        61.5%   
2                           explicit     25       13          12        69.2%   
3            explicit_mild_profanity     25        7          18        42.9%   
4          harsh_but_civil_criticism     25        3          22         0.0%   
5                    implicit_threat     24       12          12        16.7%   
6  legitimate_frustration_at_product     15        0          15            -   
7                     neutral_polite     25        3          22         0.0%   
8                 passive_aggressive     25       19           6        15.8%   
9                            sarcasm     25       15          10         6.7%   

  Prec_Toxic Recall_NonToxic Prec_NonToxic  AUC_ROC  
0      50.0%           71.4%         23.8%   0.4921  
